# STEP 1: Installation

In [ ]:
!pip install langchain-huggingface
!pip install -q --upgrade langchain langchain-core langchain-community langchain-huggingface langchain-text-splitters
!pip install -q chromadb pypdf2 sentence-transformers
!pip install -q langgraph python-multipart pillow
!pip install -q streamlit
!pip install -q langchain langchain-community langchain-huggingface chromadb langgraph pypdf2 sentence-transformers streamlit pillow python-multipart


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 56.0 MB/s eta 0:00:00


# STEP 2: Upload PDF

In [ ]:
from google.colab import files
print(" UPLOAD TechGadget PDF:")
uploaded = files.upload()

# AUTO-DETECT PDF FILENAME
pdf_files = [f for f in uploaded.keys() if f.lower().endswith('.pdf')]
if pdf_files:
    pdf_name = pdf_files[0]
    print(f" Found: {pdf_name}")
else:
    print( "No PDF uploaded!")
    raise ValueError("Upload PDF first!")

 UPLOAD TechGadget PDF:


Saving TechGadget_Refund_Return_Policy.pdf to TechGadget_Refund_Return_Policy (1).pdf
 Found: TechGadget_Refund_Return_Policy (1).pdf


# STEP 3:  Knowledge Base Pipeline

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

loader = PyPDFLoader(pdf_name)
docs = loader.load()
print(f" {len(docs)} pages loaded ")

splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f" {len(chunks)} chunks ")

embedding = HuggingFaceEmbeddings()
db = Chroma.from_documents(chunks, embedding)
print(" TechGadget DB ready! ")


 2 pages loaded ✓
 2 chunks ✓


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 TechGadget DB ready! ✓


# STEP 4:  LangGraph Workflow

In [ ]:
# =============================================================================
# LANGGRAPH + HITL (Production)
# =============================================================================
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
import operator

class State(TypedDict):
    query: str
    docs: Annotated[List[str], operator.add]
    answer: str
    confidence: float
    needs_hitl: bool

def retrieve(state: State):
    query = state["query"]
    results = db.similarity_search(query, k=2)
    docs = [doc.page_content for doc in results]
    state["docs"] = docs
    print(f" Found {len(docs)} policy sections")
    return state

def generate(state: State):
    docs_text = "\n".join(state["docs"])
    length = len(docs_text)

    if length < 100:
        state["confidence"] = 0.4
        state["answer"] = "Limited info found"
    else:
        state["confidence"] = min(0.9, 0.6 + length/500)
        state["answer"] = f"TechGadget Policy: {docs_text[:250]}..."

    state["needs_hitl"] = state["confidence"] < 0.7
    print(f" Conf: {state['confidence']:.2f}")
    return state

def hitl(state: State):
    if state["needs_hitl"]:
        print("\n HUMAN ESCALATION!")
        print(f"Query: {state['query']}")
        human = input(" Support Agent: ")
        state["answer"] = f" AGENT: {human}"
        state["confidence"] = 1.0
    return state

def output(state: State):
    print("\n" + "="*70)
    print(" TECHGADGET SUPPORT ASSISTANT")
    print(f"Customer: {state['query']}")
    print(f"Answer: {state['answer']}")
    print(f"Status: {state['confidence']:.2f} confidence")
    print("="*70)
    return state

In [ ]:
# =============================================================================
# LANGGRAPH WORKFLOW
# =============================================================================
graph = StateGraph(State)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.add_node("hitl", hitl)
graph.add_node("output", output)

def route(state: State):
    return "hitl" if state["needs_hitl"] else "output"

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "generate")
graph.add_conditional_edges("generate", route, {"hitl": "hitl", "output": "output"})
graph.add_edge("hitl", "output")
graph.add_edge("output", END)

app = graph.compile()
print(" FULL SYSTEM READY!")

 FULL SYSTEM READY!


# SREP 5: Chat

In [ ]:
# =============================================================================
# PRODUCTION CHAT
# =============================================================================
print("\n TechGadget Bot Online!")
print(" refund? return window? damaged? contact? quit")

while True:
    q = input("\n User: ").strip()
    if q.lower() in ['quit', 'exit']:
        print("\n Exiting..")
        break

    result = app.invoke({"query": q})


 TechGadget Bot Online!
 refund? return window? damaged? contact? quit

 User: refund?
 Found 2 policy sections
 Conf: 0.90

 TECHGADGET SUPPORT ASSISTANT
Customer: refund?
Answer: TechGadget Policy: Refund and Return Policy - TechGadget Retail Pvt. Ltd.
Effective Date
April 23, 2026
Return Window
- Customers may return items within 14 days of delivery.
- Returns requested after 14 days will not be accepted.
Eligible Items
- Unused, unopened prod...
Status: 0.90 confidence

 User: contact?
 Found 2 policy sections
 Conf: 0.90

 TECHGADGET SUPPORT ASSISTANT
Customer: contact?
Answer: TechGadget Policy: - Contact customer support with photos and order details.
- Replacement or full refund will be provided after verification.
Customer Contact Details
TechGadget Retail Pvt. Ltd.
Email: support@techgadgetretail.com
Phone: +91-98765-43210
Address: Plot ...
Status: 0.90 confidence

 User: exit

 Exiting..
